# Optuna AdamW Hyperparameter Search

Search AdamW-centered hyperparameters for two workloads:

- **Protein pretrain** from `config/train.yaml`
- **Protein completion** / profile-aware span completion from `config/profile_span_completion.yaml`

The search follows the small-LLM training pattern from `rasbt/LLMs-from-scratch`: AdamW, validation loss as the objective, short proxy runs, gradient clipping, and a compact search space around learning rate, weight decay, batch shape, context length, gradient accumulation, and evaluation cadence. Run short trials first, then copy the best YAML into a longer training run.

In [1]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import shutil
from pathlib import Path
from typing import Any

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import optuna
import torch
import yaml

from libs.notebook_runtime import bootstrap_notebook
from libs.core.pretrain.protein_lm.trainer import ProteinPretrainTrainer
from libs.instruction.trainer import InstructionTrainer, build_instruction_training_config
from libs.notebook_runtime import apply_instruction_notebook_overrides

RUNTIME = bootstrap_notebook(globals().get("__file__", Path.cwd()))
REPO_DIR = Path(RUNTIME["repo_dir"]).resolve()
print(json.dumps(RUNTIME, indent=2))

c:\Users\Admin\Desktop\MDNAC\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "repo_dir": "C:\\Users\\Admin\\Desktop\\MDNAC",
  "platform": "Windows",
  "platform_name": "Windows",
  "is_colab": false,
  "is_notebook": true,
  "python": "3.11.15",
  "cuda_available": true,
  "cuda_device_count": 1
}


## Search Settings

Use small proxy runs for Optuna. Each trial derives its own `max_steps` from an equal-compute budget (`PRETRAIN_TOKEN_BUDGET` / `COMPLETION_RECORD_BUDGET`, clamped to `[TRIAL_MIN_STEPS, TRIAL_MAX_STEPS]`) so trials are compared at matched data, not matched step counts. Trials use the same warmup + cosine schedule as real training (`warmup_ratio` is searched). Increase `N_TRIALS_*` and the budgets after the pipeline is confirmed on your machine, then copy the best YAML into a longer run.

In [3]:
# Base configs
PRETRAIN_BASE_CONFIG = Path(os.environ.get("MDNAC_OPTUNA_PRETRAIN_CONFIG", REPO_DIR / "config" / "train.16gb.yaml")).expanduser()
COMPLETION_BASE_CONFIG = Path(os.environ.get("MDNAC_OPTUNA_COMPLETION_CONFIG", REPO_DIR / "config" / "profile_span_completion.yaml")).expanduser()

# 16GB machines need a deliberately narrow search space. Set
# MDNAC_OPTUNA_LOW_MEMORY=0 to restore the broader exploratory search.
LOW_MEMORY_PRESET = os.environ.get("MDNAC_OPTUNA_LOW_MEMORY", "1").strip().lower() not in {"0", "false", "no", "off"}
OPTUNA_STUDY_SUFFIX = os.environ.get("MDNAC_OPTUNA_STUDY_SUFFIX", "16gb" if LOW_MEMORY_PRESET else "full")
TARGET_VRAM_GB = float(os.environ.get("MDNAC_OPTUNA_TARGET_VRAM_GB", "12" if LOW_MEMORY_PRESET else "14"))
OPTUNA_N_JOBS = int(os.environ.get("MDNAC_OPTUNA_N_JOBS", "1"))
SHUFFLE_BUFFER_SIZE = int(os.environ.get("MDNAC_OPTUNA_SHUFFLE_BUFFER_SIZE", "512" if LOW_MEMORY_PRESET else "8192"))
PROFILE_SAMPLE_SIZE = int(os.environ.get("MDNAC_OPTUNA_PROFILE_SAMPLE_SIZE", "5000" if LOW_MEMORY_PRESET else "100000"))

PRETRAIN_BATCH_CHOICES = [1, 2] if LOW_MEMORY_PRESET else [1, 2, 4, 8]
PRETRAIN_CONTEXT_CHOICES = [256, 384, 512] if LOW_MEMORY_PRESET else [256, 384, 512, 768, 1024]
COMPLETION_BATCH_CHOICES = [1, 2] if LOW_MEMORY_PRESET else [1, 2, 4, 8]
GRAD_ACCUM_CHOICES = [8, 16, 32] if LOW_MEMORY_PRESET else [1, 2, 4, 8, 16]
EVAL_FREQ_CHOICES = [0, 100] if LOW_MEMORY_PRESET else [25, 50, 100]

# Study outputs
STUDY_ROOT = Path(os.environ.get("MDNAC_OPTUNA_DIR", REPO_DIR / "data" / "optuna" / "adamw_search")).expanduser().resolve()
STORAGE_URL = f"sqlite:///{(STUDY_ROOT / 'optuna_studies.db').as_posix()}"

# Trial budgets. Keep these short for search, then retrain with the winning YAML.
N_TRIALS_PRETRAIN = int(os.environ.get("MDNAC_OPTUNA_PRETRAIN_TRIALS", "6" if LOW_MEMORY_PRESET else "12"))
N_TRIALS_COMPLETION = int(os.environ.get("MDNAC_OPTUNA_COMPLETION_TRIALS", "6" if LOW_MEMORY_PRESET else "12"))

# Equal-compute budgets. Each trial derives its own max_steps so it trains on
# roughly the SAME number of tokens (pretrain) / records (completion). Without
# this, a fixed max_steps lets large-batch / long-context trials win the search
# simply by consuming more data per step, not because their LR/wd is better.
PRETRAIN_TOKEN_BUDGET = int(os.environ.get("MDNAC_OPTUNA_PRETRAIN_TOKEN_BUDGET", "500000" if LOW_MEMORY_PRESET else "2000000"))
COMPLETION_RECORD_BUDGET = int(os.environ.get("MDNAC_OPTUNA_COMPLETION_RECORD_BUDGET", "5000" if LOW_MEMORY_PRESET else "20000"))
TRIAL_MIN_STEPS = int(os.environ.get("MDNAC_OPTUNA_TRIAL_MIN_STEPS", "20" if LOW_MEMORY_PRESET else "60"))
TRIAL_MAX_STEPS = int(os.environ.get("MDNAC_OPTUNA_TRIAL_MAX_STEPS", "120" if LOW_MEMORY_PRESET else "400"))
EVAL_BATCHES = int(os.environ.get("MDNAC_OPTUNA_EVAL_BATCHES", "1" if LOW_MEMORY_PRESET else "8"))
SEED = int(os.environ.get("MDNAC_OPTUNA_SEED", "123"))

STUDY_ROOT.mkdir(parents=True, exist_ok=True)
print("low_memory_preset:", LOW_MEMORY_PRESET)
print("study_suffix:", OPTUNA_STUDY_SUFFIX)
print("target_vram_gb:", TARGET_VRAM_GB)
print("pretrain_batch_choices:", PRETRAIN_BATCH_CHOICES)
print("pretrain_context_choices:", PRETRAIN_CONTEXT_CHOICES)
print("study_root:", STUDY_ROOT)
print("storage:", STORAGE_URL)

low_memory_preset: True
study_suffix: 16gb
target_vram_gb: 12.0
pretrain_batch_choices: [1, 2]
pretrain_context_choices: [256, 384, 512]
study_root: C:\Users\Admin\Desktop\MDNAC\data\optuna\adamw_search
storage: sqlite:///C:/Users/Admin/Desktop/MDNAC/data/optuna/adamw_search/optuna_studies.db


## Helpers

In [4]:
def load_yaml(path: Path) -> dict[str, Any]:
    resolved = path if path.is_absolute() else (REPO_DIR / path).resolve()
    data = yaml.safe_load(resolved.read_text(encoding="utf-8")) or {}
    if not isinstance(data, dict):
        raise ValueError(f"Expected YAML mapping: {resolved}")
    return data


def deep_update(base: dict[str, Any], updates: dict[str, Any]) -> dict[str, Any]:
    result = copy.deepcopy(base)
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update(result[key], value)
        else:
            result[key] = value
    return result


def dump_trial_yaml(config: dict[str, Any], path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")
    return path


def cleanup_torch() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def is_memory_error(exc: BaseException) -> bool:
    oom_type = getattr(torch, "OutOfMemoryError", None)
    if oom_type is not None and isinstance(exc, oom_type):
        return True
    if isinstance(exc, torch.cuda.OutOfMemoryError):
        return True
    message = str(exc).lower()
    return "out of memory" in message or "vram preflight" in message or "cuda oom" in message


def best_loss_from_result(result) -> float:
    candidates = [getattr(result, "best_loss", None), getattr(result, "best_val_loss", None), getattr(result, "final_val_loss", None)]
    for value in candidates:
        if value is not None and math.isfinite(float(value)):
            return float(value)
    raise optuna.TrialPruned("Trial did not produce a finite validation loss.")


def suggest_adamw_common(trial: optuna.Trial, *, prefix: str = "") -> dict[str, Any]:
    return {
        f"{prefix}learning_rate": trial.suggest_float(f"{prefix}learning_rate", 1e-5, 8e-4, log=True),
        f"{prefix}weight_decay": trial.suggest_float(f"{prefix}weight_decay", 0.0, 0.2),
        f"{prefix}gradient_accumulation_steps": trial.suggest_categorical(
            f"{prefix}gradient_accumulation_steps", GRAD_ACCUM_CHOICES
        ),
        # Warmup is searched as a fraction of each trial's own step budget so the
        # proxy uses the SAME warmup + cosine schedule the real training run uses.
        f"{prefix}warmup_ratio": trial.suggest_float(f"{prefix}warmup_ratio", 0.0, 0.1),
    }


def steps_for_budget(budget_units: int, units_per_step: int, *, min_steps: int, max_steps: int) -> int:
    """Translate an equal-compute budget into a per-trial optimizer-step count."""
    if units_per_step <= 0:
        return max_steps
    steps = round(budget_units / units_per_step)
    return int(max(min_steps, min(max_steps, steps)))


def write_best_summary(study: optuna.Study, output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = {
        "study_name": study.study_name,
        "best_value": study.best_value,
        "best_params": study.best_params,
        "best_trial_number": study.best_trial.number,
        "datetime_start": str(study.best_trial.datetime_start),
        "datetime_complete": str(study.best_trial.datetime_complete),
    }
    (output_dir / "best_trial.json").write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

## Protein Pretrain Objective

In [5]:
def build_pretrain_trial_config(trial: optuna.Trial) -> tuple[dict[str, Any], Path]:
    base = load_yaml(PRETRAIN_BASE_CONFIG)
    params = suggest_adamw_common(trial)
    batch_size = trial.suggest_categorical("batch_size", PRETRAIN_BATCH_CHOICES)
    context_length = trial.suggest_categorical("context_length", PRETRAIN_CONTEXT_CHOICES)
    eval_freq = trial.suggest_categorical("eval_freq", EVAL_FREQ_CHOICES)

    grad_accum = params["gradient_accumulation_steps"]
    tokens_per_step = batch_size * context_length * grad_accum
    trial_max_steps = steps_for_budget(
        PRETRAIN_TOKEN_BUDGET, tokens_per_step, min_steps=TRIAL_MIN_STEPS, max_steps=TRIAL_MAX_STEPS
    )

    model_updates = {"context_length": context_length, "stride": max(1, context_length // 2)}

    trial_dir = STUDY_ROOT / "pretrain" / f"trial_{trial.number:04d}"
    config = deep_update(
        base,
        {
            "mode": {"name": "train_from_scratch", "resume_if_available": False},
            "data": {
                "batch_size": batch_size,
                "num_workers": 0,
                "pin_memory": False,
                "shuffle_examples": True,
                "shuffle_buffer_size": SHUFFLE_BUFFER_SIZE,
            },
            "model": model_updates,
            "training": {
                "max_steps": trial_max_steps,
                "num_epochs": 1,
                "gradient_accumulation_steps": params["gradient_accumulation_steps"],
                "eval_freq": eval_freq,
                "eval_batches": EVAL_BATCHES,
                "save_every_steps": None,
                "save_last": False,
                "save_best": True,
                "save_final": False,
                "grad_clip_norm": 1.0,
            },
            "optimizer": {
                "type": "adamw",
                "learning_rate": params["learning_rate"],
                "weight_decay": params["weight_decay"],
                "fused": "auto",
                "lr_scheduler": "cosine",
                "warmup_ratio": params["warmup_ratio"],
                "min_lr_ratio": 0.1,
                "lr_decay_steps": trial_max_steps,
            },
            "runtime": {"multi_gpu_mode": "none", "preflight_vram_check": True, "target_vram_gb": TARGET_VRAM_GB, "seed": SEED},
            "resume": {
                "restore_optimizer_state": False,
                "checkpoint_path": str(trial_dir / "checkpoint_best.pt"),
                "output_checkpoint_path": str(trial_dir / "checkpoint_best.pt"),
                "best_checkpoint_path": str(trial_dir / "checkpoint_best.pt"),
                "final_checkpoint_path": str(trial_dir / "checkpoint_final.pt"),
                "resume_state_path": str(trial_dir / "resume_state.json"),
            },
            "paths": {
                "checkpoint_dir": str(trial_dir),
                "resume_state_path": str(trial_dir / "resume_state.json"),
                "metrics_history_path": str(trial_dir / "metrics_history.jsonl"),
                "training_config_snapshot_path": str(trial_dir / "training_config.snapshot.json"),
            },
        },
    )
    return config, trial_dir


def pretrain_objective(trial: optuna.Trial) -> float:
    torch.manual_seed(SEED + trial.number)
    config, trial_dir = build_pretrain_trial_config(trial)
    config_path = dump_trial_yaml(config, trial_dir / "trial_config.yaml")
    trainer = None
    result = None
    try:
        trainer = ProteinPretrainTrainer(REPO_DIR, config_path=config_path)
        result = trainer.train()
        objective = best_loss_from_result(result)
        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("config_path", str(config_path))
        trial.set_user_attr("global_step", int(result.global_step))
        trial.set_user_attr("tokens_seen", int(result.tokens_seen))
        trial.set_user_attr("token_budget", PRETRAIN_TOKEN_BUDGET)
        return objective
    except RuntimeError as exc:
        if is_memory_error(exc):
            raise optuna.TrialPruned("memory budget exceeded") from exc
        raise
    finally:
        del result
        del trainer
        cleanup_torch()

In [ ]:
pretrain_study = optuna.create_study(
    study_name=f"mdnac_adamw_pretrain_{OPTUNA_STUDY_SUFFIX}",
    direction="minimize",
    storage=STORAGE_URL,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    # The objective returns only a single final value (the blocking trainer does
    # not report intermediate eval losses to Optuna), so MedianPruner could never
    # prune. NopPruner makes that explicit instead of implying live pruning.
    pruner=optuna.pruners.NopPruner(),
)
pretrain_study.optimize(pretrain_objective, n_trials=N_TRIALS_PRETRAIN, n_jobs=OPTUNA_N_JOBS, gc_after_trial=True)
write_best_summary(pretrain_study, STUDY_ROOT / "pretrain")

[I 2026-06-12 14:17:04,098] A new study created in RDB with name: mdnac_adamw_pretrain_16gb


🚀 Training mode: train_from_scratch
📦 [1/6] Loading tokenizer...
✅ Tokenizer loaded — vocab_size=256
🧠 [2/6] Building model...
The MDC fast path is unavailable (missing optional libraries: causal-conv1d, flash-linear-attention). Falling back to the torch implementation. The fallback uses fp32 computation (2x VRAM for activations).
✅ Model built — 280,021,632 parameters
⚙️  [3/6] Preparing runtime...


c:\Users\Admin\Desktop\MDNAC\.venv\Lib\site-packages\torch\nn\modules\module.py:1370: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:39.)
  return t.to(


✅ Device: cuda | Distributed: False
🔧 [4/6] Creating optimizer...
✅ Optimizer ready
🔍 Running VRAM preflight check (target=12.0GB)...
✅ Preflight passed — peak=2.44GB / target=12.0GB
📂 [5/6] Building data loaders...
✅ Data loaders ready
🏋️ [6/6] Starting training loop...
📉 LR schedule: warmup=3 steps | cosine decay | min_lr_ratio=0.1 | start_lr=7.05e-05
📈 Epoch 1/1 | step=0/61 | tokens=0


## Protein Completion Objective

In [ ]:
def build_completion_trial_config(trial: optuna.Trial) -> tuple[dict[str, Any], Path]:
    base = load_yaml(COMPLETION_BASE_CONFIG)
    params = suggest_adamw_common(trial, prefix="completion_")
    batch_size = trial.suggest_categorical("completion_batch_size", COMPLETION_BATCH_CHOICES)
    eval_freq = trial.suggest_categorical("completion_eval_freq", EVAL_FREQ_CHOICES)

    grad_accum = params["completion_gradient_accumulation_steps"]
    trial_max_steps = steps_for_budget(
        COMPLETION_RECORD_BUDGET, batch_size * grad_accum, min_steps=TRIAL_MIN_STEPS, max_steps=TRIAL_MAX_STEPS
    )

    trial_dir = STUDY_ROOT / "protein_completion" / f"trial_{trial.number:04d}"
    config = deep_update(
        base,
        {
            "data": {
                "batch_size": batch_size,
                "num_workers": 0,
                "pin_memory": False,
                "shuffle_buffer_size": SHUFFLE_BUFFER_SIZE,
                "count_splits_before_training": False,
            },
            "artifacts": {"profile_sample_size": PROFILE_SAMPLE_SIZE},
            "training": {
                "max_steps": trial_max_steps,
                "num_epochs": 1,
                "gradient_accumulation_steps": params["completion_gradient_accumulation_steps"],
                "eval_freq": eval_freq,
                "eval_batches": EVAL_BATCHES,
                "save_every_steps": None,
                "save_last": False,
                "save_best": True,
                "save_final": False,
                "grad_clip_norm": 1.0,
            },
            "optimizer": {
                "type": "adamw",
                "learning_rate": params["completion_learning_rate"],
                "weight_decay": params["completion_weight_decay"],
                "fused": "auto",
                "lr_scheduler": "cosine",
                "warmup_ratio": params["completion_warmup_ratio"],
                "min_lr_ratio": 0.1,
                "lr_decay_steps": trial_max_steps,
            },
            "runtime": {"multi_gpu_mode": "none", "seed": SEED + trial.number},
            "resume": {"resume_if_available": False, "restore_optimizer_state": False},
            "paths": {"checkpoint_dir": str(trial_dir), "output_dir": str(trial_dir)},
        },
    )
    return config, trial_dir


def completion_objective(trial: optuna.Trial) -> float:
    torch.manual_seed(SEED + trial.number)
    config, trial_dir = build_completion_trial_config(trial)
    config_path = dump_trial_yaml(config, trial_dir / "trial_config.yaml")
    trainer = None
    result = None
    try:
        training_config = build_instruction_training_config(REPO_DIR, config)
        training_config = apply_instruction_notebook_overrides(training_config)
        trainer = InstructionTrainer(training_config)
        result = trainer.train()
        objective = best_loss_from_result(result)
        trial.set_user_attr("trial_dir", str(trial_dir))
        trial.set_user_attr("config_path", str(config_path))
        trial.set_user_attr("global_step", int(result.global_step))
        trial.set_user_attr("tokens_seen", int(result.tokens_seen))
        return objective
    except RuntimeError as exc:
        if is_memory_error(exc):
            raise optuna.TrialPruned("memory budget exceeded") from exc
        raise
    finally:
        del result
        del trainer
        cleanup_torch()

In [ ]:
completion_study = optuna.create_study(
    study_name=f"mdnac_adamw_protein_completion_{OPTUNA_STUDY_SUFFIX}",
    direction="minimize",
    storage=STORAGE_URL,
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED),
    # The objective returns only a single final value (the blocking trainer does
    # not report intermediate eval losses to Optuna), so MedianPruner could never
    # prune. NopPruner makes that explicit instead of implying live pruning.
    pruner=optuna.pruners.NopPruner(),
)
completion_study.optimize(completion_objective, n_trials=N_TRIALS_COMPLETION, n_jobs=OPTUNA_N_JOBS, gc_after_trial=True)
write_best_summary(completion_study, STUDY_ROOT / "protein_completion")

## Export Best YAMLs

These cells copy the best trial YAMLs to stable files you can use for real training runs.

In [ ]:
def export_best_config(study: optuna.Study, target_path: Path) -> Path:
    config_path = Path(study.best_trial.user_attrs["config_path"])
    target_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(config_path, target_path)
    return target_path

pretrain_best_yaml = export_best_config(pretrain_study, STUDY_ROOT / "best_pretrain_adamw.yaml")
completion_best_yaml = export_best_config(completion_study, STUDY_ROOT / "best_protein_completion_adamw.yaml")
print("pretrain_best_yaml:", pretrain_best_yaml)
print("completion_best_yaml:", completion_best_yaml)